In [4]:
#Cell 1 - Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')

BASE_DIR = Path(r'C:\Users\bisu2\Desktop\churn-prediction')

print("All libraries imported successfully.")

All libraries imported successfully.


In [5]:
#Cell 2 - Load Raw Data
df = pd.read_csv(BASE_DIR / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
#Cell 3 - Drop customerID
#customerID is just a unique identifier
#It has ZERO predictive value for churn 
#Like removing a student's roll number before predicting their grade

df = df.drop(columns=['customerID'])

print(f"Shape after dropping customerID: {df.shape}")
print(f"customerID dropped")

Shape after dropping customerID: (7043, 20)
customerID dropped


In [7]:
#Cell 4 - Fix TotalCharges 
#Remember from EDA - TotalCharges has 11 blank values
#These are new customers with 0 tenure 
#So we fill missing TotalCharges with 0

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Missing TotalCharges before fix: {df['TotalCharges'].isnull().sum()}")

#Fill with 0 because these are brand new customers
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"Missing TotalCharges after fix: {df['TotalCharges'].isnull().sum()}")
print(f"TotalCharges fixed.")

Missing TotalCharges before fix: 11
Missing TotalCharges after fix: 0
TotalCharges fixed.


In [8]:
#Cell 5 - Encode Target Variable
#Convert Churn: Yes -> 1, No -> 0
#This is called Binary Encoding

df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn value counts after encoding:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.1f}%")
print("Target encoded")

Churn value counts after encoding:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate: 26.5%
Target encoded


In [10]:
#Cell 6 - Identify Column Types
#We need to treat numerical and categorical columns differently
#Numerical -> Scale (StandardScaler)
#Categorical -> Encode (OneHotEncoder)

#Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

#Identify column types 
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nCategorical columns ({len(categorical_cols)})")
print(categorical_cols)

print(f"\nTarget  (y) shape: {y.shape}")
print(f"Feature  (X) shape: {X.shape}")

Numerical columns (4):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns (15)
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Target  (y) shape: (7043,)
Feature  (X) shape: (7043, 19)


In [11]:
#Cell 7 - Train Test Split
#Split BEFORE any preprocessing to avoid data leakage
#Data leakage = when test data influences training = cheating
#Think of it like: don't let exam answers leak into study material

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set : {X_train.shape}")
print(f"Test set: {y_test.shape}")
print(f"\nChurn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test : {y_test.mean()*100:.1f}%")
print("\nTrain/Test split done.")

Training set : (5634, 19)
Test set: (1409,)

Churn rate in train: 26.5%
Churn rate in test : 26.5%

Train/Test split done.


In [12]:
#Cell 8 - Build Preprocessing Pipeline
#Instead of manually transforming columns one by one
#We build a Pipeline - professional, clean, no data leakage

#Numerical pipeline -> just scale the numbers
numerical_pipeline = Pipeline(steps=[
    ('Scaler', StandardScaler())
])

#Categorical pipeline -> encode text to numbers
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

#Combine both pipelines into one ColumnTransformer
preprocessor =  ColumnTransformer(transformers= [
    ('num' , numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline built")
print("\nPipeline structure")
print(f"   Numerical columns -> StandardScaler")
print(f" Categorical columns -> OneHotEncoder")

Preprocessing pipeline built

Pipeline structure
   Numerical columns -> StandardScaler
 Categorical columns -> OneHotEncoder


In [13]:
#Cell 9 - Apply SMOTE
#SMOTE = Synthetic Minority Oversmpling Technique
#Problem: 73.5% Stay vs 26.5% Churn -> imbalanced
#Think of it like : photocopying minority class examples with slight variations
#IMPORTANT: Apply SMOTE only on training data NEVER on test data

#First transform the data

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Before SMOTE - Train shape: {X_train_transformed.shape}")
print(f"Before SMOTE - Churn count: {y_train.value_counts().to_dict()}")

#Apply SMOTE 
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_transformed, y_train
)

print(f"\nAfter SMOTE - Train shape: {X_train_resampled.shape}")
print(f"After SMOTE - Churn count: {pd.Series(y_train_resampled).value_counts().to_dict()}")
print("\nSMOTE applied")

Before SMOTE - Train shape: (5634, 45)
Before SMOTE - Churn count: {0: 4139, 1: 1495}

After SMOTE - Train shape: (8278, 45)
After SMOTE - Churn count: {0: 4139, 1: 4139}

SMOTE applied


In [15]:
#Cell 10 - Save Processed Data
processed_dir = BASE_DIR / 'data' / 'processed'

#Save transformed arrays
np.save(processed_dir / 'X_train.npy', X_train_resampled)
np.save(processed_dir / 'X_test.npy', X_test_transformed)
np.save(processed_dir / 'y_train.npy', y_train_resampled)
np.save(processed_dir / 'y_test.npy', y_test.values)

#Save preprocessor pipeline for use in API later
joblib.dump(preprocessor, BASE_DIR / 'models' / 'preprocessor.pkl')

#Save column names for reference
joblib.dump(numerical_cols, BASE_DIR / 'models' / 'numerical_cols.pkl')
joblib.dump(categorical_cols, BASE_DIR / 'models' / 'categorical_cols.pkl')

print("Files Saved:")
print(f"data/processed/X_train.npy")
print(f"data/processed/X_test.npy")
print(f"data/processed/y_train.npy")
print(f"data/processed/y_test.npy")
print(f"models/preprocessor.pkl")
print(f"models/numerical_cols.pkl")
print(f"models/categorical_cols.pkl")

Files Saved:
data/processed/X_train.npy
data/processed/X_test.npy
data/processed/y_train.npy
data/processed/y_test.npy
models/preprocessor.pkl
models/numerical_cols.pkl
models/categorical_cols.pkl
